# CLEIDS-Edge — Notebook 03: Training CLEIDS-Edge

Trains the hybrid CNN-LSTM architecture (`build_cleids_edge`, Notebook 02) on all five benchmark datasets (NSL-KDD, CICIDS2017, UNSW-NB15, TON_IoT, IoT-23) for both Binary (benign/attack) and Multi-class classification tasks. IoT-23 uses a reservoir-sampled 2,000,000-row subset (~162x smaller than the full ~325.3M-row dataset) — see Notebook 01's IoT-23 section and `CLEIDS_PROJECT_BRIEF.md` for the full rationale.

**GPU runtime (L4) is required for this notebook.** The GPU-verification cell enforces GPU visibility and halts execution if no GPU is available.

**Run this notebook directly in Colab's browser UI** (colab.research.google.com), not through a proxied kernel connection (e.g. Antigravity's Colab MCP proxy). `google.colab.userdata.get()` (the `GITHUB_TOKEN` secret lookup in the repo-setup cell) requires the actual Colab frontend to be present for its permission handshake — over a proxied connection it times out (`TimeoutException: ... Secrets can only be fetched when running from the Colab UI`). If you must use a proxied connection, set `os.environ["GITHUB_TOKEN"]` manually for that session instead (and never commit that cell with a real token in it).

**This notebook cannot be executed by Claude Code** — there is no tool available to run cells against a remote Colab kernel, and the local machine has no GPU. No metric here is real until you run it yourself.

**Data bridge**: Notebook 01 ran locally, so `data/processed/*.npz` exists only on the local machine, not in Colab's `/content`. Upload your local `data/processed/` folder once to Google Drive at `MyDrive/CLEIDS_Edge/data_processed/` (via drive.google.com or the Drive desktop app), then the data-bridge cell below copies it into Colab's local disk. (An earlier localtunnel-based bridge measured only ~65KB/s — 3+ hours for 772MB — and was abandoned in favor of this Drive-based approach.)

Independent per-dataset sections (same convention as Notebook 01). See `CLEIDS_PROJECT_BRIEF.md` for overall thesis pipeline context.

## 1. Repo setup (clone/pull + auth)

In [1]:
import os
import subprocess

REPO_URL = "https://github.com/NehlTech/CLEIDS-Edge.git"
REPO_DIR = "/content/CLEIDS-Edge"

GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as e:
    print(f"[DEBUG] userdata.get('GITHUB_TOKEN') raised {type(e).__name__}: {e}")
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError("GITHUB_TOKEN not found. Add it to Colab secrets (key icon, left sidebar).")

AUTH_REMOTE = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", AUTH_REMOTE, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", AUTH_REMOTE], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


Working directory: /content/CLEIDS-Edge


## 2. Google Drive mount (source for processed data; backup target for model checkpoints/results/figures)

In [2]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/CLEIDS_Edge"
DRIVE_MODELS = os.path.join(DRIVE_ROOT, "models")
DRIVE_RESULTS = os.path.join(DRIVE_ROOT, "results")
DRIVE_FIGURES = os.path.join(DRIVE_ROOT, "figures")
for d in (DRIVE_MODELS, DRIVE_RESULTS, DRIVE_FIGURES):
    os.makedirs(d, exist_ok=True)
print("Drive ready at:", DRIVE_ROOT, "-- backup target in case hours of GPU compute finish before a successful git push.")
DRIVE_DATA_PROCESSED = os.path.join(DRIVE_ROOT, "data_processed")


Mounted at /content/drive
Drive ready at: /content/drive/MyDrive/CLEIDS_Edge -- backup target in case hours of GPU compute finish before a successful git push.


## 3. Data bridge — copy processed splits from Google Drive

In [3]:
import shutil

DATASETS = ["nsl-kdd", "cicids2017", "unsw-nb15", "ton-iot", "iot-23"]

if not os.path.isdir(DRIVE_DATA_PROCESSED):
    raise RuntimeError(
        f"{DRIVE_DATA_PROCESSED} not found. Upload your local data/processed/ folder to Google "
        f"Drive at MyDrive/CLEIDS_Edge/data_processed/ first (via drive.google.com or the Drive "
        f"desktop app), then re-run this cell."
    )

for name in DATASETS:
    src_dir = os.path.join(DRIVE_DATA_PROCESSED, name)
    dst_dir = f"data/processed/{name}"
    os.makedirs(dst_dir, exist_ok=True)
    for fn in ["train.npz", "val.npz", "test.npz", "label_classes.json", "feature_names.json"]:
        src = os.path.join(src_dir, fn)
        dst = os.path.join(dst_dir, fn)
        if not os.path.exists(src):
            raise RuntimeError(
                f"{src} not found on Drive. The upload for '{name}' looks incomplete -- "
                f"re-upload data/processed/{name}/ to Drive and re-run this cell."
            )
        if os.path.exists(dst):
            print(f"[skip] {dst} already present")
            continue
        shutil.copy2(src, dst)
        print(f"[copied] {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

manifest_dst = "data/processed/preprocessing_manifest.json"
if not os.path.exists(manifest_dst):
    manifest_src = os.path.join(DRIVE_DATA_PROCESSED, "preprocessing_manifest.json")
    if not os.path.exists(manifest_src):
        raise RuntimeError(f"{manifest_src} not found on Drive.")
    shutil.copy2(manifest_src, manifest_dst)

print("\nAll processed data copied from Drive.")


[copied] data/processed/nsl-kdd/train.npz (84.0 MB)
[copied] data/processed/nsl-kdd/val.npz (0.7 MB)
[copied] data/processed/nsl-kdd/test.npz (1.4 MB)
[copied] data/processed/nsl-kdd/label_classes.json (0.0 MB)
[copied] data/processed/nsl-kdd/feature_names.json (0.0 MB)
[copied] data/processed/cicids2017/train.npz (460.0 MB)
[copied] data/processed/cicids2017/val.npz (67.5 MB)
[copied] data/processed/cicids2017/test.npz (67.7 MB)
[copied] data/processed/cicids2017/label_classes.json (0.0 MB)
[copied] data/processed/cicids2017/feature_names.json (0.0 MB)
[copied] data/processed/unsw-nb15/train.npz (64.6 MB)
[copied] data/processed/unsw-nb15/val.npz (2.3 MB)
[copied] data/processed/unsw-nb15/test.npz (8.3 MB)
[copied] data/processed/unsw-nb15/label_classes.json (0.0 MB)
[copied] data/processed/unsw-nb15/feature_names.json (0.0 MB)
[copied] data/processed/ton-iot/train.npz (48.0 MB)
[copied] data/processed/ton-iot/val.npz (2.0 MB)
[copied] data/processed/ton-iot/test.npz (2.0 MB)
[copied]

## 4. Setup & GPU Verification

In [ ]:
import sys
import time
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc as sklearn_auc, precision_recall_curve, average_precision_score
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

tf.random.set_seed(42)
np.random.seed(42)

# GPU Enforcement -- this notebook must not silently fall back to CPU
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError(
        "CRITICAL ERROR: No GPU runtime detected! Notebook 03 requires a GPU (L4). "
        "Please switch your runtime to GPU in Colab (Runtime -> Change runtime type -> GPU) "
        "before running this notebook."
    )
print(f"[HARDWARE OK] GPU visible: {gpus}")
print("TensorFlow version:", tf.__version__)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from models import build_cleids_edge

for d in ["models", "results", "figures"]:
    os.makedirs(d, exist_ok=True)

with open("data/processed/preprocessing_manifest.json") as f:
    prep_manifest = json.load(f)
print("Loaded preprocessing manifest. Ready datasets:", list(prep_manifest["datasets"].keys()))

[HARDWARE OK] GPU visible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow version: 2.20.0


## 5. Evaluation & Plotting Utilities

`validate_against_manifest` loads `{train,val,test}.npz` and asserts row counts match Notebook 01's `preprocessing_manifest.json` exactly — **stops with a clear error on any mismatch**, never proceeds silently. `train_and_evaluate_model` builds via `build_cleids_edge`, trains with the specified callbacks, evaluates once on test, and flags a warning if accuracy lands at/near the majority-class baseline (non-convergence is reported, never hidden). AUC failures are reported as `None`, never silently replaced with a placeholder value.

In [ ]:
def validate_against_manifest(dataset_name, train_data, val_data, test_data):
    expected = prep_manifest["datasets"][dataset_name]["shapes"]
    actual = {"train": train_data["X_cnn"].shape[0], "val": val_data["X_cnn"].shape[0], "test": test_data["X_cnn"].shape[0]}
    for split, actual_n in actual.items():
        expected_n = expected[split][0]
        if actual_n != expected_n:
            raise RuntimeError(
                f"[{dataset_name}] {split} row count mismatch: Notebook 01 manifest says {expected_n:,}, "
                f"loaded {actual_n:,}. Stopping rather than proceeding on mismatched data."
            )
    print(f"[VALIDATE] {dataset_name}: train={actual['train']:,} val={actual['val']:,} test={actual['test']:,} "
          f"-- matches Notebook 01 manifest exactly")


def calculate_fpr(y_true, y_pred, binary=True):
    """Calculate False Positive Rate (FPR = FP / (FP + TN)); macro-averaged one-vs-rest for multiclass."""
    cm = confusion_matrix(y_true, y_pred)
    if binary:
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    else:
        fprs = []
        for i in range(len(cm)):
            tp = cm[i, i]
            fp = cm[:, i].sum() - tp
            fn = cm[i, :].sum() - tp
            tn = cm.sum() - (tp + fp + fn)
            class_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            fprs.append(class_fpr)
        fpr = float(np.mean(fprs))
    return float(fpr)


def plot_confusion_matrix(y_true, y_pred, class_names, save_path, title):
    """Plot and save 300 DPI confusion matrix figure."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(8, 6) if len(class_names) <= 10 else (12, 10), dpi=300)
    sns.heatmap(
        cm, annot=(len(class_names) <= 15), fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names
    )
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved confusion matrix plot to {save_path}")


def plot_training_curves(history, save_path, title):
    """Plot and save 300 DPI training and validation loss/accuracy curves."""
    plt.figure(figsize=(12, 5), dpi=300)
    plt.subplot(1, 2, 1)
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.title(f"{title} - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.subplot(1, 2, 2)
    acc_key = "accuracy" if "accuracy" in history.history else "acc"
    val_acc_key = "val_" + acc_key
    plt.plot(history.history[acc_key], label="Train Accuracy")
    plt.plot(history.history[val_acc_key], label="Val Accuracy")
    plt.title(f"{title} - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved training curves to {save_path}")

def plot_roc_curve(y_true_cls, y_score, class_names, save_path, title, binary):
    """ROC curve: binary -- single curve + AUC. Multiclass -- one-vs-rest per
    class plus a macro-average curve, since a single AUC number doesn't show
    which classes the model actually confuses."""
    plt.figure(figsize=(7, 6), dpi=300)
    if binary:
        fpr, tpr, _ = roc_curve(y_true_cls, y_score)
        roc_auc = sklearn_auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"ROC (AUC = {roc_auc:.4f})")
    else:
        y_true_bin = tf.keras.utils.to_categorical(y_true_cls, num_classes=len(class_names))
        fprs, tprs = [], []
        for i, name in enumerate(class_names):
            if y_true_bin[:, i].sum() == 0:
                continue  # class absent from this test set -- no meaningful curve to plot
            fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
            auc_i = sklearn_auc(fpr_i, tpr_i)
            plt.plot(fpr_i, tpr_i, alpha=0.5, linewidth=1, label=f"{name} (AUC={auc_i:.3f})")
            fprs.append(fpr_i)
            tprs.append(tpr_i)
        all_fpr = np.unique(np.concatenate(fprs))
        mean_tpr = np.zeros_like(all_fpr)
        for fpr_i, tpr_i in zip(fprs, tprs):
            mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
        mean_tpr /= len(fprs)
        macro_auc = sklearn_auc(all_fpr, mean_tpr)
        plt.plot(all_fpr, mean_tpr, color="black", linewidth=2.5, label=f"macro-average (AUC={macro_auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(fontsize=7 if not binary else 10, loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved ROC curve to {save_path}")


def plot_pr_curve(y_true_cls, y_score, class_names, save_path, title, binary):
    """Precision-Recall curve -- often more informative than ROC on these
    imbalanced datasets, since a rare-attack ROC can look deceptively good
    while precision collapses. Same one-vs-rest treatment for multiclass."""
    plt.figure(figsize=(7, 6), dpi=300)
    if binary:
        prec, rec, _ = precision_recall_curve(y_true_cls, y_score)
        ap = average_precision_score(y_true_cls, y_score)
        plt.plot(rec, prec, label=f"PR (AP = {ap:.4f})")
    else:
        y_true_bin = tf.keras.utils.to_categorical(y_true_cls, num_classes=len(class_names))
        aps = []
        for i, name in enumerate(class_names):
            if y_true_bin[:, i].sum() == 0:
                continue
            prec_i, rec_i, _ = precision_recall_curve(y_true_bin[:, i], y_score[:, i])
            ap_i = average_precision_score(y_true_bin[:, i], y_score[:, i])
            plt.plot(rec_i, prec_i, alpha=0.5, linewidth=1, label=f"{name} (AP={ap_i:.3f})")
            aps.append(ap_i)
        plt.plot([], [], color="black", linewidth=2.5, label=f"macro-average AP={np.mean(aps):.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend(fontsize=7 if not binary else 10, loc="lower left")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved Precision-Recall curve to {save_path}")


def plot_per_class_metrics_bar(per_class_report, save_path, title):
    """Bar chart of precision/recall/F1 per class -- visual companion to the
    classification_report table, multiclass only."""
    rows = {k: v for k, v in per_class_report.items() if k not in ("accuracy", "macro avg", "weighted avg")}
    df = pd.DataFrame(rows).transpose()[["precision", "recall", "f1-score"]]
    df.plot(kind="bar", figsize=(max(8, len(df) * 0.6), 5))
    plt.title(title)
    plt.ylabel("Score")
    plt.ylim(0, 1.05)
    plt.xticks(rotation=45, ha="right")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved per-class metrics bar chart to {save_path}")


def plot_embeddings_tsne_pca(model, X_test, y_true_cls, class_names, save_path, title, max_points=5000, seed=42):
    """PCA (linear baseline) and t-SNE (nonlinear) projections of the trained
    model's penultimate-layer ('dense_1') embeddings on a subsample of the
    test set, colored by true class -- shows class separability the raw
    metrics don't. Subsampled since t-SNE on the full test set (up to
    300k rows for IoT-23) is impractically slow."""
    embedding_model = tf.keras.Model(inputs=model.input, outputs=model.get_layer("dense_1").output)
    rng = np.random.default_rng(seed)
    n = X_test.shape[0]
    idx = rng.choice(n, size=min(max_points, n), replace=False)
    embeddings = embedding_model.predict(X_test[idx], batch_size=512, verbose=0)
    labels_sub = y_true_cls[idx]

    pca_proj = PCA(n_components=2, random_state=seed).fit_transform(embeddings)
    tsne_proj = TSNE(n_components=2, random_state=seed, init="pca", perplexity=30).fit_transform(embeddings)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=300)
    cmap = "tab10" if len(class_names) <= 10 else "tab20"
    scatter = None
    for ax, proj, name in [(axes[0], pca_proj, "PCA"), (axes[1], tsne_proj, "t-SNE")]:
        scatter = ax.scatter(proj[:, 0], proj[:, 1], c=labels_sub, cmap=cmap, s=5, alpha=0.6)
        ax.set_title(f"{name} projection")
    handles, _ = scatter.legend_elements(num=len(class_names))
    fig.legend(handles, class_names, loc="upper center", ncol=min(len(class_names), 6), bbox_to_anchor=(0.5, 1.08))
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved t-SNE/PCA embedding plot (n={len(idx):,} sampled points) to {save_path}")


def plot_cross_dataset_summary(main_results, save_path):
    """Cross-dataset bar chart (accuracy/F1/AUC, binary vs multiclass) built
    from this notebook's own main_results -- the 'at a glance' figure that
    usually opens a Results chapter. Does NOT include baseline models
    (Notebook 04) or latency/model-size (Notebook 06) -- those stay scoped
    to their own notebooks."""
    rows = []
    for ds_name, tasks in main_results.items():
        for task_name, m in tasks.items():
            rows.append({
                "dataset": ds_name, "task": task_name, "accuracy": m["accuracy"],
                "f1": m["f1"], "auc": m["auc"] if m["auc"] is not None else np.nan,
            })
    df = pd.DataFrame(rows)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=300)
    for ax, metric in zip(axes, ["accuracy", "f1", "auc"]):
        pivot = df.pivot(index="dataset", columns="task", values=metric)
        pivot.plot(kind="bar", ax=ax, rot=45)
        ax.set_title(metric.upper())
        ax.set_ylim(0, 1.05)
        ax.legend(loc="lower right")
    fig.suptitle("CLEIDS-Edge -- Cross-Dataset Summary (own results, not baseline comparison)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved cross-dataset summary chart to {save_path}")


In [ ]:
def train_and_evaluate_model(dataset_name, binary=True, batch_size=256):
    """Train CLEIDS-Edge on given dataset split and evaluate on held-out test set."""
    task_str = "binary" if binary else "multiclass"
    print("\n" + "=" * 70)
    print(f"[TRAINING] {dataset_name} ({task_str.upper()})")
    print("=" * 70)

    data_dir = os.path.join("data/processed", dataset_name)
    train_data = np.load(os.path.join(data_dir, "train.npz"))
    val_data = np.load(os.path.join(data_dir, "val.npz"))
    test_data = np.load(os.path.join(data_dir, "test.npz"))

    validate_against_manifest(dataset_name, train_data, val_data, test_data)

    X_train, X_val, X_test = train_data["X_cnn"], val_data["X_cnn"], test_data["X_cnn"]

    with open(os.path.join(data_dir, "label_classes.json")) as f:
        label_info = json.load(f)
    class_names = label_info["classes"]
    num_classes = len(class_names)
    input_dim = X_train.shape[1]

    if binary:
        y_train = train_data["y_bin"].astype(np.float32)
        y_val = val_data["y_bin"].astype(np.float32)
        y_test = test_data["y_bin"].astype(np.float32)
    else:
        y_train_idx = train_data["y_multi"].astype(np.int32)
        y_val_idx = val_data["y_multi"].astype(np.int32)
        y_test_idx = test_data["y_multi"].astype(np.int32)
        y_train = tf.keras.utils.to_categorical(y_train_idx, num_classes=num_classes)
        y_val = tf.keras.utils.to_categorical(y_val_idx, num_classes=num_classes)
        y_test = tf.keras.utils.to_categorical(y_test_idx, num_classes=num_classes)

    model = build_cleids_edge(input_dim=input_dim, num_classes=num_classes, binary=binary)
    ckpt_path = f"models/cleids_edge_{dataset_name}_{task_str}.keras"

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
        tf.keras.callbacks.ModelCheckpoint(filepath=ckpt_path, monitor="val_loss", save_best_only=True, verbose=0),
    ]

    start_time = time.time()
    try:
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1,
        )
    except tf.errors.ResourceExhaustedError:
        print(f"[OOM RECOVERY] OOM with batch_size={batch_size}. Retrying with batch_size=128...")
        batch_size = 128
        model = build_cleids_edge(input_dim=input_dim, num_classes=num_classes, binary=binary)
        start_time = time.time()
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1,
        )

    train_time_sec = float(time.time() - start_time)
    epochs_run = len(history.history["loss"])
    early_stopped = epochs_run < 50
    print(f"Training complete in {train_time_sec:.2f}s ({train_time_sec/60:.1f} min), "
          f"{epochs_run}/50 epochs, batch_size_used={batch_size}, "
          f"{'early-stopped' if early_stopped else 'ran full 50 epochs'}.")

    if os.path.exists(ckpt_path):
        model = tf.keras.models.load_model(ckpt_path)

    y_pred_raw = model.predict(X_test, batch_size=512, verbose=0)
    if binary:
        y_pred_prob = y_pred_raw.ravel()
        y_pred_cls = (y_pred_prob >= 0.5).astype(int)
        y_true_cls = y_test.astype(int)
        try:
            auc = float(roc_auc_score(y_true_cls, y_pred_prob))
        except ValueError as e:
            auc = None
            print(f"[WARN] AUC could not be computed: {e}")
        acc = float(accuracy_score(y_true_cls, y_pred_cls))
        prec = float(precision_score(y_true_cls, y_pred_cls, zero_division=0))
        rec = float(recall_score(y_true_cls, y_pred_cls, zero_division=0))
        f1 = float(f1_score(y_true_cls, y_pred_cls, zero_division=0))
        fpr = calculate_fpr(y_true_cls, y_pred_cls, binary=True)
        plot_cls_names = ["Benign", "Attack"]
        per_class_report = None
    else:
        y_pred_cls = np.argmax(y_pred_raw, axis=1)
        y_true_cls = y_test_idx
        # sklearn's roc_auc_score(multi_class="ovr", average="macro") does NOT raise when a
        # class has zero real occurrences in y_test -- it silently emits an
        # UndefinedMetricWarning and returns nan for that class, which then poisons the macro
        # average into an uninformative nan without ever hitting our except ValueError below
        # (confirmed: NSL-KDD's official KDDTest+ has zero 'spy'/'warezclient' rows -- a real
        # property of that fixed official file, not a pipeline bug -- and this silently produced
        # AUC=nan on the first real training run). Computing per-class AUC explicitly instead,
        # excluding classes absent from this test split from the average, with the exclusion
        # printed rather than hidden.
        present_classes = [i for i in range(num_classes) if y_test[:, i].any()]
        absent_classes = [class_names[i] for i in range(num_classes) if i not in present_classes]
        if absent_classes:
            print(f"[WARN] {len(absent_classes)} class(es) have zero real occurrences in this "
                  f"test split (support=0) -- excluded from macro AUC rather than silently "
                  f"propagating sklearn's NaN for an undefined per-class AUC: {absent_classes}")
        try:
            per_class_auc = [roc_auc_score(y_test[:, i], y_pred_raw[:, i]) for i in present_classes]
            auc = float(np.mean(per_class_auc))
        except ValueError as e:
            auc = None
            print(f"[WARN] Multiclass AUC could not be computed: {e}")
        acc = float(accuracy_score(y_true_cls, y_pred_cls))
        prec = float(precision_score(y_true_cls, y_pred_cls, average="macro", zero_division=0))
        rec = float(recall_score(y_true_cls, y_pred_cls, average="macro", zero_division=0))
        f1 = float(f1_score(y_true_cls, y_pred_cls, average="macro", zero_division=0))
        fpr = calculate_fpr(y_true_cls, y_pred_cls, binary=False)
        plot_cls_names = class_names
        # labels=range(num_classes) is required: without it, sklearn infers the label set from
        # whatever appears in y_true/y_pred, which can be fewer than num_classes (e.g. NSL-KDD's
        # ultra-rare spy/perl/phf may be entirely absent from the test set) -- that mismatch
        # against a fixed-length target_names raises ValueError.
        per_class_report = classification_report(
            y_true_cls, y_pred_cls, labels=list(range(num_classes)),
            target_names=class_names, output_dict=True, zero_division=0,
        )
        print(f"\n[{dataset_name} MULTI-CLASS PER-CLASS METRICS]")
        print(pd.DataFrame(per_class_report).transpose().to_string())

    fig_title = f"CLEIDS-Edge {dataset_name} ({task_str.upper()})"
    fig_cm_path = f"figures/confusion_matrix_{dataset_name}_{task_str}.png"
    fig_curve_path = f"figures/training_curve_{dataset_name}_{task_str}.png"
    fig_roc_path = f"figures/roc_curve_{dataset_name}_{task_str}.png"
    fig_pr_path = f"figures/pr_curve_{dataset_name}_{task_str}.png"
    fig_embed_path = f"figures/embeddings_{dataset_name}_{task_str}.png"

    plot_confusion_matrix(y_true_cls, y_pred_cls, plot_cls_names, fig_cm_path, fig_title)
    plot_training_curves(history, fig_curve_path, fig_title)
    roc_pr_score = y_pred_prob if binary else y_pred_raw
    roc_pr_class_names = plot_cls_names if not binary else None
    plot_roc_curve(y_true_cls, roc_pr_score, roc_pr_class_names, fig_roc_path, fig_title, binary=binary)
    plot_pr_curve(y_true_cls, roc_pr_score, roc_pr_class_names, fig_pr_path, fig_title, binary=binary)
    if per_class_report is not None:
        fig_perclass_path = f"figures/per_class_metrics_{dataset_name}_{task_str}.png"
        plot_per_class_metrics_bar(per_class_report, fig_perclass_path, fig_title)
    plot_embeddings_tsne_pca(model, X_test, y_true_cls, plot_cls_names, fig_embed_path, fig_title)

    metrics = {
        "accuracy": acc, "precision": prec, "recall": rec, "f1": f1,
        "auc": auc, "fpr": fpr, "train_time_sec": round(train_time_sec, 2),
        "epochs_run": epochs_run, "batch_size_used": batch_size, "checkpoint_path": ckpt_path,
    }
    if per_class_report is not None:
        metrics["per_class_report"] = per_class_report

    auc_str = f"{auc:.4f}" if auc is not None else "N/A"
    print(f"[{dataset_name} {task_str.upper()} RESULTS] Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} "
          f"F1={f1:.4f} AUC={auc_str} FPR={fpr:.4f}")

    majority_baseline = np.bincount(y_true_cls.astype(int)).max() / len(y_true_cls)
    if acc <= majority_baseline + 0.01:
        print(f"[WARNING] Accuracy ({acc:.4f}) is at/near the majority-class baseline "
              f"({majority_baseline:.4f}) -- this model may not have learned anything beyond "
              f"predicting the majority class. Reporting as-is, not hiding it.")

    return metrics

## 6. NSL-KDD Training

`num_classes=40` includes NSL-KDD's 17 test-only novel attack classes (a documented dataset property, not a bug) — expect near-zero recall specifically on those in the per-class table, not full-dataset non-convergence.

**After this section, read the time-estimate cell below before continuing** — per the run request, total time for all 8 runs should be estimated here before committing to the rest.

In [ ]:
t0 = time.time()
nsl_bin_metrics = train_and_evaluate_model("nsl-kdd", binary=True)
nsl_mc_metrics = train_and_evaluate_model("nsl-kdd", binary=False)
nsl_elapsed = time.time() - t0

print(f"\n[ESTIMATE] nsl-kdd (binary+multiclass) took {nsl_elapsed/60:.1f} min total.")
print(f"[ESTIMATE] Rough projection for the remaining 3 datasets x 2 tasks: "
      f"~{nsl_elapsed/60*3:.0f}-{nsl_elapsed/60*5:.0f} min more (varies with dataset size).")
print("[ESTIMATE] STOP AND REVIEW: NSL-KDD's train set (1,212,169 rows) is the second-smallest of "
      "the four post-SMOTE; CICIDS2017 (2,275,692 rows) and TON_IoT (2,022,420 rows) are roughly "
      "double, so the real total will likely run higher than this naive projection for those two. "
      "If the projected total exceeds a few hours, decide now whether to let Colab run unattended "
      "(e.g. overnight) or step in, per the run instructions.")

## 7. CICIDS2017 Training

Largest training set of the four (2,275,692 rows post-capped-SMOTE) -- expect this to be the slowest.

In [ ]:
cic_bin_metrics = train_and_evaluate_model("cicids2017", binary=True)
cic_mc_metrics = train_and_evaluate_model("cicids2017", binary=False)

## 8. UNSW-NB15 Training

Smallest training set of the four (504,000 rows) -- expect this to be the fastest.

In [ ]:
unsw_bin_metrics = train_and_evaluate_model("unsw-nb15", binary=True)
unsw_mc_metrics = train_and_evaluate_model("unsw-nb15", binary=False)

## 9. TON_IoT Training

Second-largest training set (2,022,420 rows post-SMOTE).

In [ ]:
ton_bin_metrics = train_and_evaluate_model("ton-iot", binary=True)
ton_mc_metrics = train_and_evaluate_model("ton-iot", binary=False)

## 10. IoT-23 Training

Uses the reservoir-sampled subset (2,000,000 of ~325.3M rows) prepared in Notebook 01 — see that notebook's IoT-23 section for the full sampling/label-merge/SMOTE methodology.

In [ ]:
iot23_bin_metrics = train_and_evaluate_model("iot-23", binary=True)
iot23_mc_metrics = train_and_evaluate_model("iot-23", binary=False)


## 11. Consolidated Results & Artifact Saving

In [ ]:
main_results = {
    "nsl-kdd": {"binary": nsl_bin_metrics, "multiclass": nsl_mc_metrics},
    "cicids2017": {"binary": cic_bin_metrics, "multiclass": cic_mc_metrics},
    "unsw-nb15": {"binary": unsw_bin_metrics, "multiclass": unsw_mc_metrics},
    "ton-iot": {"binary": ton_bin_metrics, "multiclass": ton_mc_metrics},
    "iot-23": {"binary": iot23_bin_metrics, "multiclass": iot23_mc_metrics},
}

results_path = "results/main_results.json"
with open(results_path, "w") as f:
    json.dump(main_results, f, indent=2)
print(f"Wrote consolidated results to {results_path}")

plot_cross_dataset_summary(main_results, "figures/cross_dataset_summary.png")

## 12. Backup to Drive + push to GitHub

In [ ]:
# Drive backup -- models/ included, not just results/figures, since these represent hours of
# GPU compute that would otherwise be lost if the Colab session ends before a successful push.
shutil.copy2(results_path, os.path.join(DRIVE_RESULTS, "main_results.json"))
for fn in os.listdir("figures"):
    shutil.copy2(os.path.join("figures", fn), os.path.join(DRIVE_FIGURES, fn))
for fn in os.listdir("models"):
    shutil.copy2(os.path.join("models", fn), os.path.join(DRIVE_MODELS, fn))
print("Drive backup of models/, figures/, results/ complete at", DRIVE_ROOT)

# All checkpoint files are tiny for this architecture (~483KB each, ~4MB for 8 models --
# param count is independent of input_dim, see Notebook 02), so plain git works with no LFS
# needed -- verified below rather than assumed.
large_files = [os.path.join("models", fn) for fn in os.listdir("models")
               if os.path.getsize(os.path.join("models", fn)) > 50 * 1024 * 1024]
if large_files:
    print(f"[WARN] {len(large_files)} checkpoint file(s) exceed 50MB -- excluded from git push, "
          f"Drive backup above is the only copy: {large_files}")
else:
    print("All checkpoint files are under 50MB -- safe to push to git directly.")

model_files = [f"models/{fn}" for fn in os.listdir("models") if f"models/{fn}" not in large_files]
figure_files = [f"figures/{fn}" for fn in os.listdir("figures")]
add_paths = ["notebooks/03_Training_CLEIDS_Edge.ipynb", results_path] + model_files + figure_files

subprocess.run(["git", "-C", REPO_DIR, "add"] + add_paths, check=True)
commit_res = subprocess.run(
    ["git", "-C", REPO_DIR, "commit", "-m", "Notebook 03: train CLEIDS-Edge on 4 datasets (binary + multiclass)"],
    capture_output=True, text=True,
)
print(commit_res.stdout, commit_res.stderr)
if commit_res.returncode == 0:
    subprocess.run(["git", "-C", REPO_DIR, "push", "origin", "HEAD"], check=True)
    print("Pushed to GitHub.")
else:
    print("Nothing new to commit (or commit failed) -- see output above.")

## 13. Final Consolidated Summary

Paste this cell's output back for review before Notebook 04 (baselines).

In [ ]:
print("\n" + "=" * 95)
print("CLEIDS-Edge -- Notebook 03 Headline Results Summary")
print("=" * 95)
header = (f"{'Dataset':<12} | {'Task':<10} | {'Accuracy':<8} | {'Precision':<9} | {'Recall':<8} | "
          f"{'F1-Score':<8} | {'AUC':<7} | {'FPR':<7} | {'Time (s)':<8} | {'Epochs':<6}")
print(header)
print("-" * 95)
for ds_name, tasks in main_results.items():
    for task_name, m in tasks.items():
        auc_str = f"{m['auc']:.4f}" if m["auc"] is not None else "N/A"
        row = (f"{ds_name:<12} | {task_name:<10} | {m['accuracy']:<8.4f} | {m['precision']:<9.4f} | "
               f"{m['recall']:<8.4f} | {m['f1']:<8.4f} | {auc_str:<7} | {m['fpr']:<7.4f} | "
               f"{m['train_time_sec']:<8.1f} | {m['epochs_run']:<6}")
        print(row)
print("=" * 95)
print("Full details (per-class reports, checkpoint paths) are in results/main_results.json.")